# ARQWELIA Lot 2 — SDXL Inpainting on a free GPU (benchmark only)

**WARNING**
- GPU availability is NOT guaranteed on free environments.
- A free environment is NOT appropriate for Production.
- Delete temporary files after the session.
- Do NOT use a real user photo during Phase 0A (synthetic benchmark images only).
- Do NOT expose ComfyUI on the Internet; no public tunnel is included.
- Do NOT store any DeepSeek API key in this notebook.
- The image + mask stay local; nothing is sent to a remote provider.

## 1. Verify CUDA GPU presence

In [ ]:
import subprocess
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True, timeout=20)
    print(out.stdout or "NO GPU / nvidia-smi not found")
except Exception as exc:
    print("GPU check failed:", exc)


## 2. Install locked dependencies

Versions are pinned for reproducibility. Requires a CUDA-capable runtime (PyTorch).

In [ ]:
import sys
!{sys.executable} -m pip install --quiet torch==2.4.1 diffusers==0.31.0 transformers==4.44.2 accelerate==0.34.2 pillow==10.4.0
print("dependencies installed")

## 3. Load SDXL Inpainting from its official repository

In [ ]:
import torch
from diffusers import StableDiffusionXLInpaintPipeline

model_id = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"
pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipe.to("cuda" if torch.cuda.is_available() else "cpu")
print("pipeline loaded")

## 4. Load source image + mask

Paths are local. The mask must be grayscale (black=preserve, white=modify) and same size as the image.

In [ ]:
from PIL import Image

SOURCE_PATH = "dataset/photos/synthetic01.png"
MASK_PATH = "dataset/masks/synthetic01-pool-mask.png"

image = Image.open(SOURCE_PATH).convert("RGB")
mask = Image.open(MASK_PATH).convert("L")
print("image size:", image.size, "| mask size:", mask.size)
assert mask.size == image.size, "mask and image must be the same size"

## 5. Use a VisualBrief JSON

In [ ]:
import json

visual_brief = {
    "version": "arqwelia-visual-brief-v1",
    "concept": "A",
    "sceneType": "residential_garden_pool_inpainting",
    "pool": {
        "shape": "rectangular",
        "estimatedDimensions": "8x4m",
        "placement": "central_open_lawn",
        "orientation": "parallel_to_house",
    },
    "preserve": ["house_architecture", "camera_perspective", "boundary_fences", "mature_trees", "unmasked_pixels"],
    "add": ["realistic_in_ground_pool", "natural_stone_coping", "subtle_mediterranean_landscaping"],
    "negative": ["people", "text", "logo", "house_distortion", "extra_buildings", "duplicate_pool", "floating_objects", "unrealistic_reflections"],
    "inpaintingPrompt": "Photorealistic in-ground swimming pool added to a residential front garden. Pool shape: rectangular. Pool dimensions: 8x4m. Garden style: mediterranean. Coping: natural_stone. Terrace: natural_stone_patio. Budget range: medium. Preserve the house, fences, existing trees and the exact camera perspective. Only modify the masked area; keep every unmasked pixel unchanged. Natural lighting, no people, no text, no logos, realistic water reflections.",
    "negativePrompt": "people, faces, text, watermark, logo, distorted architecture, extra buildings, second pool, floating objects, unrealistic reflections, warped geometry, construction equipment, cartoon style, oversaturated colors",
    "recommended": {"steps": 25, "cfg": 7, "strength": 0.82, "seed": 42},
}
print(json.dumps(visual_brief, indent=2)[:600])

## 6. Run ONE manual generation

In [ ]:
import torch
from diffusers.utils import load_image

generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(visual_brief["recommended"]["seed"])
result = pipe(
    prompt=visual_brief["inpaintingPrompt"],
    negative_prompt=visual_brief["negativePrompt"],
    image=image,
    mask_image=mask,
    width=1024,
    height=1024,
    num_inference_steps=visual_brief["recommended"]["steps"],
    guidance_scale=visual_brief["recommended"]["cfg"],
    strength=visual_brief["recommended"]["strength"],
    generator=generator,
)
print("generation done")

## 7. Save the image

In [ ]:
output_image = result.images[0]
output_image.save("benchmark-out/deepseek-comfyui-poc/notebook-sdxl-result.png")
print("saved")

## 8. Print technical metadata

In [ ]:
print("seed:", visual_brief["recommended"]["seed"])
print("steps:", visual_brief["recommended"]["steps"])
print("cfg:", visual_brief["recommended"]["cfg"])
print("strength:", visual_brief["recommended"]["strength"])
print("output size:", output_image.size)
print("model:", model_id)

## 9. Stop

Generation complete. Delete temporary files and free the GPU after the session.